In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy

2025-12-06 17:58:27.700292: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-06 17:58:27.700329: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-06 17:58:27.701531: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-06 17:58:27.709235: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-06 17:58:28.671599: W tensorflow/compiler/tf2

In [2]:
batch_size = 256
learning_rate = 0.001

In [3]:
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [4]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.c = None
    def upload(self, delta_ws, delta_cs):
        for v, dw in zip(self.model.variables, delta_ws):
            v.assign_add(dw)
        for c_global, dc in zip(self.c, delta_cs):
            c_global.assign_add(dc)
        return self.model, self.c
    def download(self):
        return self.model, self.c
    def initModel(self, x):
        self.model(x)
        if self.c is None:
            self.c = [tf.Variable(tf.zeros_like(v), trainable=False) for v in self.model.trainable_variables]

In [5]:
def valiAll():
    m, _ = ps.download()
    model = copy.deepcopy(m)
    y_v_p = model(X_v)
    va_mse = tf.reduce_mean(tf.square(y_v_p - y_v))
    va_rmse = tf.sqrt(va_mse)
    va_mae = tf.reduce_mean(tf.abs(y_v_p - y_v))
    va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - y_v)) / tf.reduce_sum(tf.square(y_v - tf.reduce_mean(y_v)))
    print("mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
    r2sv.append(va_r2.numpy())

In [6]:
class Node:
    def __init__(self, dsName,freq):
        self.model = MLP()
        self.freq = freq
        dataset = pd.read_csv(dsName, encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X = dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y = dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=23000)
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
        self.c_i = None
    def train(self, num_epochs):
        for round_idx in range(num_epochs):
            # 1. 从服务器下载当前全局模型参数 w_t 和全局控制变量 c
            global_model, c_global = ps.download()
            self.model = copy.deepcopy(global_model)

            # 初始化本地 c_i
            if self.c_i is None:
                self.c_i = [tf.zeros_like(v) for v in self.model.variables]

            # 2. 保存本轮开始时的参数 w_t
            w_before = [tf.identity(v) for v in self.model.variables]

            step = 0
            last_tr_mse = None
            last_tr_rmse = None
            last_tr_mae = None
            last_tr_r2 = None

            # 3. 本地用修正梯度做 K 步更新
            for X, y in self.dataset_train:

                with tf.GradientTape() as tape:
                    y_pred = self.model(X)
                    tr_mse = tf.reduce_mean(tf.square(y_pred - y))

                grads = tape.gradient(tr_mse, self.model.variables)

                # 修正梯度：g - c_i + c
                corrected_grads = [
                    g - ci + cg
                    for g, ci, cg in zip(grads, self.c_i, c_global)
                ]

                # 用简单 SGD 更新本地模型参数
                for v, g_corr in zip(self.model.variables, corrected_grads):
                    v.assign_sub(learning_rate * g_corr)

                # 记录最后一个 batch 的指标（方便打印）
                last_tr_mse = tr_mse
                last_tr_rmse = tf.sqrt(tr_mse)
                last_tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
                last_tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(
                    tf.square(y - tf.reduce_mean(y))
                )
                step += 1
            # 4. 本地更新完成后，计算 Δw_i
            w_after = [tf.identity(v) for v in self.model.variables]
            delta_w = [
                w_a - w_b
                for w_a, w_b in zip(w_after, w_before)
            ]

            # 5. 按 SCAFFOLD 公式更新本地控制变量 c_i，并得到 Δc_i
            #    c_i_new = c_i - c + (1 / (K * η)) * (w_before - w_after)
            K = max(step, 1)  # 防止除零
            scale = 1.0 / (K * learning_rate)

            new_c_i = []
            delta_c_i = []
            for ci_old, c_g, w_b, w_a in zip(self.c_i, c_global, w_before, w_after):
                ci_new = ci_old - c_g + scale * (w_b - w_a)
                new_c_i.append(ci_new)
                delta_c_i.append(ci_new - ci_old)

            # 更新本地 c_i
            self.c_i = new_c_i

            # 6. 把 Δw_i 和 Δc_i 上传给服务器
            global_model, c_global = ps.upload(delta_w, delta_c_i)

            # 7. 打印训练信息 + 记录 r2 + 调用你的验证函数
            if last_tr_mse is not None:
                print("node:{} round:{}".format(self.freq, round_idx))
                print("train mse:{} rmse:{} mae:{} r2:{}".format(
                    last_tr_mse, last_tr_rmse, last_tr_mae, last_tr_r2
                ))
                r2s.append(last_tr_r2.numpy())

            # 用当前全局模型做验证（假设 valiAll 内部会用 ps.model 或保存好的全局模型）
            valiAll()

In [7]:
r2s = []
r2sv = []

In [8]:
test_dataset = pd.read_csv("Test.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
X_v = test_dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

In [9]:
ps = ParaServer()
ps.initModel(X_v)

2025-12-06 17:58:29.677987: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 900 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:02:00.0, compute capability: 8.9


In [10]:
nodeList = [Node('./24Train.csv', 2.4), Node('./25Train.csv', 2.5), Node('./26Train.csv', 2.6)]

In [11]:
nodeList[0].train(100)
nodeList[1].train(100)
nodeList[2].train(100)
nodeList[0].train(100)
nodeList[1].train(100)
nodeList[2].train(100)

node:2.4 round:0
train mse:0.0921449363231659 rmse:0.30355384945869446 mae:0.2447783201932907 r2:0.2384740114212036
mse:0.11210300028324127 rmse:0.3348178565502167 mae:0.27067625522613525 r2:0.07202446460723877
node:2.4 round:1
train mse:0.09327279776334763 rmse:0.30540594458580017 mae:0.2461407333612442 r2:0.2350780963897705
mse:0.10033497214317322 rmse:0.3167569637298584 mae:0.2573525905609131 r2:0.16943883895874023
node:2.4 round:2
train mse:0.08955777436494827 rmse:0.29926204681396484 mae:0.24898074567317963 r2:0.2587989568710327
mse:0.09665019065141678 rmse:0.3108861446380615 mae:0.2526063024997711 r2:0.19994103908538818
node:2.4 round:3
train mse:0.0864359512925148 rmse:0.29399991035461426 mae:0.24453644454479218 r2:0.2881878614425659
mse:0.09365439414978027 rmse:0.3060300648212433 mae:0.24874050915241241 r2:0.2247399091720581
node:2.4 round:4
train mse:0.08868527412414551 rmse:0.2978007197380066 mae:0.24488449096679688 r2:0.2595391273498535
mse:0.09274379909038544 rmse:0.3045386

In [13]:
for i in r2sv:
    print(i)

0.072024465
0.16943884
0.19994104
0.22473991
0.23227763
0.2561236
0.25843513
0.275279
0.28114396
0.28160185
0.27848774
0.2944436
0.30222887
0.29740727
0.30969632
0.3074091
0.3139035
0.31590366
0.32086927
0.3229493
0.326473
0.3171373
0.32877582
0.33180255
0.33621103
0.3245942
0.33774787
0.32027954
0.3386826
0.3384276
0.33384264
0.33586627
0.34884995
0.34720725
0.3485855
0.3399182
0.35343856
0.35439295
0.34931076
0.35191584
0.35775125
0.35303408
0.35487103
0.3268327
0.36099845
0.36059707
0.360034
0.35587835
0.36242974
0.36514616
0.3572855
0.35451633
0.36459213
0.36696565
0.36493003
0.36232942
0.36968565
0.37202388
0.36817098
0.36505705
0.36894864
0.37579554
0.37514335
0.36120075
0.37062645
0.3605597
0.37525266
0.37363392
0.36459905
0.37784505
0.37613374
0.3643576
0.38300937
0.37896937
0.37728733
0.37650067
0.38082463
0.38174945
0.3830858
0.36125165
0.37847716
0.37633198
0.3821308
0.3869697
0.38499916
0.3865388
0.38036728
0.39127994
0.38048393
0.38584673
0.38405955
0.38750142
0.3882473
0.